In [1]:
from pathlib import Path
import re
import math

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D

from Bio import AlignIO
from pymsaviz import MsaViz
import session_info
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


# ----------------------------------------------------------------
# USER SETTINGS
# Edit this section only when changing dataset / histone / tissue.
# ----------------------------------------------------------------

dataset_tag = "midgut_H4" # (select liver  aor midgut and select H3 or H4)

csv_file = Path(
    "20260420_TroutGut_peptides_ions_raw_ab_rank3_msqrob-input.csv" # "260418_TroutLiver_H3H4_raw_ab_msqrob_input.csv" for liver or "20260420_TroutGut_peptides_ions_raw_ab_rank3_msqrob-input.csv" for midgut 
)

msa_file = Path("H4_alignement.fas")

protein_name = "Histone H4"
site_prefix = "H4"

# Must match the exact reference FASTA identifier
reference_id = "sp|P62805|H4" # or "sp|P62805|H4" for H4 or "sp|P68431|H31" for H3

mod_col = "Variable modifications ([position] description)"

# H4-specific Mascot artefact. Use set() for datasets without it.
excluded_peptides = set()

# Sequence display settings
wrap_length = 80
residue_font_size = 14
x_unit_size = 0.24
y_unit_size = 0.36
ptm_stack_spacing = 1.15

# Optional manual corrections for non-standard FASTA headers.
# Example: {"A0A060W2E3": "Trout"}
species_id_overrides = {}


# ----------------------------------------------------------------
# PTM classes supported globally across all datasets
# ----------------------------------------------------------------

ptm_order = (
    "Ac",
    "Me1",
    "Me2",
    "Me3",
    "Bu",
    "Prop",
    "Cr",
    "La",
    "Ox",
    "Deam",
    "Fo",
)

# Choose the PTMs shown in this specific figure.
# Default: show all recognised PTMs.
plot_ptm_types = set(ptm_order)

# Example: final Midgut H4 feature set only
# plot_ptm_types = {"Ac", "La", "Me2", "Me3"}

marker_styles = {
    "Ac":   {"marker": "*", "color": "black", "size": 11},
    "Me1":  {"marker": "o", "color": "black", "size": 11},
    "Me2":  {"marker": "s", "color": "black", "size": 11},
    "Me3":  {"marker": "^", "color": "black", "size": 11},
    "Bu":   {"marker": "h", "color": "black", "size": 11},
    "Prop": {"marker": "v", "color": "black", "size": 11},
    "Cr":   {"marker": "P", "color": "black", "size": 11},
    "La":   {"marker": "D", "color": "black", "size": 11},
    "Ox":   {"marker": "X", "color": "black", "size": 11},
    "Deam": {"marker": "p", "color": "black", "size": 11},
    "Fo":   {"marker": "<", "color": "black", "size": 11},
}

ptm_labels = {
    "Ac": "Acetylation",
    "Me1": "Monomethylation",
    "Me2": "Dimethylation",
    "Me3": "Trimethylation",
    "Bu": "Butyrylation",
    "Prop": "Propionylation",
    "Cr": "Crotonylation",
    "La": "Lactylation",
    "Ox": "Oxidation",
    "Deam": "Deamidation",
    "Fo": "Formylation",
}

# ----------------------------------------------------------------
# Species configuration
# Define all accepted organisms, display labels, and colours here.
# ----------------------------------------------------------------

species_config = {
    "Homo sapiens": {
        "label": "Human",
        "color": "#4E79A7",
    },
    "Mus musculus": {
        "label": "Mouse",
        "color": "#F28E2B",
    },
    "Oncorhynchus mykiss": {
        "label": "Rainbow trout",
        "color": "#59A14F",
    },
    "Saccharomyces cerevisiae": {
        "label": "S. cerevisiae",
        "color": "#B07AA1",
    },
    "Drosophila melanogaster": {
        "label": "D. melanogaster",
        "color": "#E15759",
    },
    "Xenopus laevis": {
        "label": "X. laevis",
        "color": "#76B7B2",
    },
    "Caenorhabditis elegans": {
        "label": "C. elegans",
        "color": "#FF9DA7",
    },
    "Arabidopsis thaliana": {
        "label": "A. thaliana",
        "color": "#EDC948",
    },
}

# Needed for custom trout FASTA headers without OS= annotation
custom_taxon_codes = {
    "ONCMY": "Oncorhynchus mykiss",
}

invalid_ptm_selection = set(plot_ptm_types) - set(ptm_order)

if invalid_ptm_selection:
    raise ValueError(
        "Unknown PTM classes in plot_ptm_types: "
        f"{sorted(invalid_ptm_selection)}"
    )

out_dir = Path(f"{dataset_tag}_MSA_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

In [2]:
# ----------------------------------------------------------------
# PTM parsing
# ----------------------------------------------------------------

def parse_canonical_modifications(value):
    """Extract tuples: (canonical_position, residue, raw_modification)."""

    if pd.isna(value) or not str(value).strip():
        return []

    return [
        (
            int(position),
            residue.strip().upper(),
            modification.strip()
        )
        for position, residue, modification in re.findall(
            r"\[(\d+)\]\s*\(([^)]+)\)\s*([^|;]+)",
            str(value)
        )
    ]


def normalise_ptm_type(modification_raw):
    """
    Convert raw PTM labels to explicit classes.

    Bare 'Me' is deliberately not converted to Me1.
    It becomes Me_unspecified and stops the workflow later.
    """

    match = re.match(
        r"\s*([A-Za-z0-9]+)",
        str(modification_raw)
    )

    if match is None:
        return None

    token = match.group(1).lower()

    if token == "me":
        return "Me_unspecified"

    aliases = {
        "ac": "Ac",
        "acetyl": "Ac",
        "acetylation": "Ac",

        "me1": "Me1",
        "monomethyl": "Me1",
        "monomethylation": "Me1",

        "me2": "Me2",
        "dimethyl": "Me2",
        "dimethylation": "Me2",

        "me3": "Me3",
        "trimethyl": "Me3",
        "trimethylation": "Me3",

        "bu": "Bu",
        "butyryl": "Bu",
        "butyrylation": "Bu",

        "prop": "Prop",
        "propionyl": "Prop",
        "propionylation": "Prop",

        "cr": "Cr",
        "crotonyl": "Cr",
        "crotonylation": "Cr",

        "la": "La",
        "lactyl": "La",
        "lactylation": "La",

        "ox": "Ox",
        "oxidation": "Ox",

        "deam": "Deam",
        "deamidation": "Deam",

        "fo": "Fo",
        "formyl": "Fo",
        "formylation": "Fo",
    }

    return aliases.get(token)


def short_sequence_name(record):
    """
    Keep the original FASTA identifier plus taxon code.

    Examples:
    sp|P68431|H41 HUMAN Histone H4.1 ... -> sp|P68431|H41 HUMAN
    sp|P08898|H4 CAEEL Histone H4 ...    -> sp|P08898|H4 CAEEL
    A0A060W2E3|H4 ONCMY                  -> A0A060W2E3|H4 ONCMY
    ACO07612.1|H4.5 ONCMY                -> ACO07612.1|H4.5 ONCMY
    """

    header = record.description.strip()

    # Remove UniProt metadata after the protein name
    header = header.split(" OS=", maxsplit=1)[0].strip()

    parts = header.split()

    if len(parts) < 2:
        raise ValueError(
            "Could not create compact sequence label from FASTA header:\n"
            f"{record.description}"
        )

    identifier = parts[0]
    taxon_code = parts[1].upper()

    return f"{identifier} {taxon_code}"

In [3]:
def infer_species(record):
    """
    Return the species display label defined in species_config.
    """

    header = record.description.strip()

    # Optional manual override:
    # key = exact record.id
    # value = scientific name used in species_config
    if record.id in species_id_overrides:
        scientific_name = species_id_overrides[record.id]

    else:
        # UniProt headers:
        # ... OS=Homo sapiens OX=9606 ...
        os_match = re.search(
            r"\bOS=(.+?)\s+OX=",
            header
        )

        if os_match:
            scientific_name = os_match.group(1).strip()

            # Remove strain text from the yeast entry
            if scientific_name.startswith(
                "Saccharomyces cerevisiae"
            ):
                scientific_name = "Saccharomyces cerevisiae"

        else:
            # Custom headers without OS= metadata, e.g.:
            # ACO07612.1|H4.5 ONCMY
            scientific_name = None
            header_upper = header.upper()

            for taxon_code, species_name in custom_taxon_codes.items():
                if re.search(
                    rf"(?:^|\s|_){re.escape(taxon_code.upper())}\b",
                    header_upper
                ):
                    scientific_name = species_name
                    break

    if scientific_name is None:
        raise ValueError(
            "Could not determine species from FASTA header:\n"
            f"{header}"
        )

    if scientific_name not in species_config:
        raise ValueError(
            "Species is not defined in species_config:\n"
            f"{scientific_name}\n\n"
            f"FASTA header:\n{header}"
        )

    return species_config[scientific_name]["label"]

In [4]:
df = pd.read_csv(csv_file)

required_columns = {
    "Protein",
    "Sequence",
    mod_col,
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns: {sorted(missing_columns)}"
    )

protein_df = df.loc[
    df["Protein"].eq(protein_name),
    ["Protein", "Sequence", mod_col]
].copy()

protein_df["peptide"] = (
    protein_df["Sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

protein_df = protein_df.loc[
    protein_df["peptide"].str.fullmatch(
        r"[ACDEFGHIKLMNPQRSTVWY]+",
        na=False
    )
    & ~protein_df["peptide"].isin(excluded_peptides)
].copy()

peptides = sorted(protein_df["peptide"].unique())

if not peptides:
    raise ValueError(
        f"No valid peptides retained for {protein_name}."
    )

pd.DataFrame(
    {"peptide": peptides}
).to_csv(
    out_dir / f"{dataset_tag}_identified_peptides.tsv",
    sep="\t",
    index=False
)

print(f"Rows retained: {len(protein_df)}")
print(f"Unique peptides retained: {len(peptides)}")

Rows retained: 53
Unique peptides retained: 10


In [5]:
mod_rows = []

for _, row in protein_df[
    ["peptide", mod_col]
].dropna(subset=[mod_col]).iterrows():

    for position, residue, modification_raw in parse_canonical_modifications(
        row[mod_col]
    ):
        mod_rows.append(
            {
                "peptide": row["peptide"],
                "reference_position": position,
                "reported_residue": residue,
                "modification_raw": modification_raw,
                "modification_type": normalise_ptm_type(
                    modification_raw
                ),
            }
        )

mods_all = pd.DataFrame(
    mod_rows,
    columns=[
        "peptide",
        "reference_position",
        "reported_residue",
        "modification_raw",
        "modification_type",
    ],
).drop_duplicates()

if mods_all.empty:
    raise ValueError("No PTM annotations were extracted.")

mods_all.to_csv(
    out_dir / f"{dataset_tag}_all_PTM_annotations.tsv",
    sep="\t",
    index=False
)

ptm_audit = (
    mods_all
    .assign(
        modification_type_display=lambda x: (
            x["modification_type"]
            .fillna("Unrecognised")
        )
    )
    [
        [
            "modification_raw",
            "modification_type_display",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "modification_type_display",
            "modification_raw",
        ]
    )
    .rename(
        columns={
            "modification_type_display": "modification_type"
        }
    )
    .reset_index(drop=True)
)

ptm_audit.to_csv(
    out_dir / f"{dataset_tag}_PTM_annotation_audit.tsv",
    sep="\t",
    index=False
)

display(ptm_audit)

bare_me = mods_all.loc[
    mods_all["modification_type"].eq("Me_unspecified"),
    [
        "peptide",
        "reference_position",
        "modification_raw",
    ],
].drop_duplicates()

if not bare_me.empty:
    display(bare_me)

    raise ValueError(
        "Bare 'Me' annotations were found. "
        "Rename them to explicit Me1 before running this workflow."
    )

unknown_raw_labels = sorted(
    mods_all.loc[
        mods_all["modification_type"].isna(),
        "modification_raw",
    ].unique()
)

detected_ptm_types = sorted(
    mods_all["modification_type"]
    .dropna()
    .unique()
)

print("Detected PTM classes:", detected_ptm_types)
print("Selected PTM classes:", sorted(plot_ptm_types))

if unknown_raw_labels:
    print("Unrecognised PTM labels:", unknown_raw_labels)

mods_plot = mods_all.loc[
    mods_all["modification_type"].isin(plot_ptm_types)
].copy()

if mods_plot.empty:
    raise ValueError(
        "No PTMs remain after applying plot_ptm_types."
    )

print(
    "Excluded recognised PTM classes:",
    sorted(
        set(detected_ptm_types)
        - set(plot_ptm_types)
    )
)

,modification_raw,modification_type
0,Ac,Ac
1,Bu,Bu
2,Cr,Cr
3,Deam,Deam
4,Fo,Fo
5,La,La
6,Me1,Me1
7,Me2,Me2
8,Ox,Ox


Detected PTM classes: ['Ac', 'Bu', 'Cr', 'Deam', 'Fo', 'La', 'Me1', 'Me2', 'Ox']
Selected PTM classes: ['Ac', 'Bu', 'Cr', 'Deam', 'Fo', 'La', 'Me1', 'Me2', 'Me3', 'Ox', 'Prop']
Excluded recognised PTM classes: []


In [6]:
alignment = AlignIO.read(msa_file, "fasta")

if len(alignment) == 0:
    raise ValueError("The alignment FASTA contains no sequences.")

alignment_lengths = {
    len(record.seq)
    for record in alignment
}

if len(alignment_lengths) != 1:
    raise ValueError(
        "Sequences have unequal alignment lengths. "
        "Use an aligned FASTA file."
    )

reference_record = next(
    (
        record
        for record in alignment
        if record.id == reference_id
    ),
    None
)

if reference_record is None:
    raise ValueError(
        f"Reference sequence not found: {reference_id}"
    )

gap_chars = {"-", "."}

reference_aligned = str(reference_record.seq).upper()

reference_ungapped = "".join(
    residue
    for residue in reference_aligned
    if residue not in gap_chars
)

reference_alignment_columns = [
    alignment_column
    for alignment_column, residue in enumerate(
        reference_aligned,
        start=1
    )
    if residue not in gap_chars
]

position_to_residue = dict(
    enumerate(reference_ungapped, start=1)
)

position_to_alignment_column = dict(
    enumerate(
        reference_alignment_columns,
        start=1
    )
)

mods_plot["reference_residue"] = (
    mods_plot["reference_position"]
    .map(position_to_residue)
)

mods_plot["alignment_column"] = (
    mods_plot["reference_position"]
    .map(position_to_alignment_column)
)

invalid_positions = sorted(
    mods_plot.loc[
        mods_plot["reference_residue"].isna()
        | mods_plot["alignment_column"].isna(),
        "reference_position",
    ].unique()
)

if invalid_positions:
    raise ValueError(
        "PTM coordinates outside the reference sequence: "
        f"{invalid_positions}"
    )

residue_mismatch = mods_plot.loc[
    mods_plot["reported_residue"]
    != mods_plot["reference_residue"],
    [
        "peptide",
        "reference_position",
        "reported_residue",
        "reference_residue",
        "modification_raw",
    ],
].drop_duplicates()

if not residue_mismatch.empty:
    display(residue_mismatch)

    raise ValueError(
        "At least one PTM residue does not match "
        "the selected reference sequence."
    )

mods_plot["canonical_site"] = (
    site_prefix
    + mods_plot["reference_residue"]
    + mods_plot["reference_position"].astype(str)
)

mods_plot.to_csv(
    out_dir / f"{dataset_tag}_plotted_PTM_annotations.tsv",
    sep="\t",
    index=False
)

ptm_rank = {
    ptm_type: rank
    for rank, ptm_type in enumerate(ptm_order)
}

ptm_marker_table = (
    mods_plot[
        [
            "canonical_site",
            "alignment_column",
            "modification_type",
        ]
    ]
    .drop_duplicates()
    .assign(
        ptm_rank=lambda x: (
            x["modification_type"]
            .map(ptm_rank)
        )
    )
    .sort_values(
        [
            "alignment_column",
            "ptm_rank",
        ]
    )
    .reset_index(drop=True)
)

if ptm_marker_table["ptm_rank"].isna().any():
    raise ValueError(
        "At least one selected PTM lacks "
        "a marker definition."
    )

ptm_marker_table["n_ptm_types_at_site"] = (
    ptm_marker_table
    .groupby("alignment_column")["modification_type"]
    .transform("size")
)

ptm_marker_table["stack_level"] = (
    ptm_marker_table
    .groupby("alignment_column")
    .cumcount()
)

ptm_marker_table["y_offset"] = (
    ptm_marker_table["stack_level"]
    * ptm_stack_spacing
)

ptm_marker_table.to_csv(
    out_dir / f"{dataset_tag}_plotted_PTM_sites.tsv",
    sep="\t",
    index=False
)

used_ptm_types = [
    ptm_type
    for ptm_type in ptm_order
    if ptm_type in set(
        ptm_marker_table["modification_type"]
    )
]

print("Reference sequence length:", len(reference_ungapped))
print("Alignment length:", len(reference_aligned))
print(
    "Distinct PTM residues:",
    ptm_marker_table["alignment_column"].nunique()
)
print(
    "Site × PTM combinations:",
    len(ptm_marker_table)
)

display(ptm_marker_table)


# ----------------------------------------------------------------
# Sequence species, colours, and display labels
# ----------------------------------------------------------------

sequence_species = [
    infer_species(record)
    for record in alignment
]

species_order = list(dict.fromkeys(sequence_species))

# Build colour lookup from the species_config defined in Chunk 1
species_colors = {
    config["label"]: config["color"]
    for config in species_config.values()
}

undefined_species = set(sequence_species) - set(species_colors)

if undefined_species:
    raise ValueError(
        "Species label(s) missing from species_config: "
        f"{sorted(undefined_species)}"
    )

sequence_labels = [
    short_sequence_name(record)
    for record in alignment
]

sequence_info = pd.DataFrame(
    {
        "sequence_id": [
            record.id
            for record in alignment
        ],
        "fasta_header": [
            record.description
            for record in alignment
        ],
        "species": sequence_species,
        "display_label": sequence_labels,
        "ungapped_length": [
            len(
                str(record.seq)
                .replace("-", "")
                .replace(".", "")
            )
            for record in alignment
        ],
    }
)

sequence_info.to_csv(
    out_dir / f"{dataset_tag}_alignment_sequence_info.tsv",
    sep="\t",
    index=False
)

display(sequence_info)

Reference sequence length: 102
Alignment length: 102
Distinct PTM residues: 11
Site × PTM combinations: 21


,canonical_site,alignment_column,modification_type,ptm_rank,n_ptm_types_at_site,stack_level,y_offset
0,H4K5,5,Ac,0,4,0,0.00
1,H4K5,5,Bu,4,4,1,1.15
2,H4K5,5,La,7,4,2,2.30
3,H4K5,5,Fo,10,4,3,3.45
4,H4K8,8,Ac,0,3,0,0.00
5,H4K8,8,Cr,6,3,1,1.15
6,H4K8,8,Fo,10,3,2,2.30
7,H4K12,12,Ac,0,2,0,0.00
8,H4K12,12,Cr,6,2,1,1.15
9,H4K16,16,Ac,0,4,0,0.00


,sequence_id,fasta_header,species,display_label,ungapped_length
0,sp|P62805|H4,sp|P62805|H4 HUMAN Histone H4 OS=Homo sapiens ...,Human,sp|P62805|H4 HUMAN,102
1,sp|Q99525|H4G,sp|Q99525|H4G HUMAN Histone H4-like protein ty...,Human,sp|Q99525|H4G HUMAN,97
2,sp|P62806|H4,sp|P62806|H4 MOUSE Histone H4 OS=Mus musculus ...,Mouse,sp|P62806|H4 MOUSE,102
3,sp|P84040|H4,sp|P84040|H4 DROME Histone H4 OS=Drosophila me...,D. melanogaster,sp|P84040|H4 DROME,102
4,sp|P62799|H4,sp|P62799|H4 XENLA Histone H4 OS=Xenopus laevi...,X. laevis,sp|P62799|H4 XENLA,102
5,sp|P02309|H4,sp|P02309|H4 YEAST Histone H4 OS=Saccharomyces...,S. cerevisiae,sp|P02309|H4 YEAST,102
6,sp|P62784|H4,sp|P62784|H4 CAEEL Histone H4 OS=Caenorhabditi...,C. elegans,sp|P62784|H4 CAEEL,102
7,sp|P59259|H4,sp|P59259|H4 ARATH Histone H4 OS=Arabidopsis t...,A. thaliana,sp|P59259|H4 ARATH,102
8,sp|P62797|H4,sp|P62797|H4 ONCMY,Rainbow trout,sp|P62797|H4 ONCMY,102
9,XP,XP 036830134.1|H4-like ONCMY,Rainbow trout,XP 036830134.1|H4-LIKE,102


In [7]:
def map_peptides_to_alignment(
    alignment,
    peptides
):
    """Return peptide-coloured alignment cells and peptide matches."""

    cell_to_peptides = {}
    match_rows = []

    for row_pos, record in enumerate(alignment):
        aligned_sequence = str(record.seq).upper()

        ungapped_to_column = [
            column
            for column, residue in enumerate(aligned_sequence)
            if residue not in {"-", "."}
        ]

        ungapped_sequence = (
            aligned_sequence
            .replace("-", "")
            .replace(".", "")
        )

        for peptide in peptides:
            for match in re.finditer(
                rf"(?={re.escape(peptide)})",
                ungapped_sequence
            ):
                start = match.start()

                matched_columns = ungapped_to_column[
                    start:start + len(peptide)
                ]

                for column in matched_columns:
                    cell_to_peptides.setdefault(
                        (row_pos, column),
                        set()
                    ).add(peptide)

                match_rows.append(
                    {
                        "sequence_id": record.id,
                        "row_pos": row_pos,
                        "peptide": peptide,
                        "ungapped_start": start + 1,
                        "ungapped_end": start + len(peptide),
                        "n_residues_coloured": len(
                            matched_columns
                        ),
                    }
                )

    cell_to_peptide = {
        cell: sorted(
            matched_peptides,
            key=lambda peptide: (
                -len(peptide),
                peptide,
            ),
        )[0]
        for cell, matched_peptides in cell_to_peptides.items()
    }

    match_table = pd.DataFrame(
        match_rows,
        columns=[
            "sequence_id",
            "row_pos",
            "peptide",
            "ungapped_start",
            "ungapped_end",
            "n_residues_coloured",
        ],
    )

    return cell_to_peptide, match_table


if len(peptides) <= 20:
    peptide_cmap = plt.colormaps["tab20"].resampled(
        len(peptides)
    )
else:
    peptide_cmap = plt.colormaps["turbo"].resampled(
        len(peptides)
    )

peptide_colors = {
    peptide: to_hex(
        peptide_cmap(
            index / max(1, len(peptides) - 1)
        )
    )
    for index, peptide in enumerate(peptides)
}

peptide_color_table = pd.DataFrame(
    {
        "peptide": peptides,
        "colour": [
            peptide_colors[peptide]
            for peptide in peptides
        ],
    }
)

peptide_color_table.to_csv(
    out_dir / f"{dataset_tag}_peptide_colour_key.tsv",
    sep="\t",
    index=False
)

cell_to_peptide, peptide_match_table = (
    map_peptides_to_alignment(
        alignment,
        peptides
    )
)

peptide_match_table.to_csv(
    out_dir / f"{dataset_tag}_peptide_alignment_matches.tsv",
    sep="\t",
    index=False
)

matched_peptides = set(
    peptide_match_table["peptide"]
)

unmatched_peptides = sorted(
    set(peptides)
    - matched_peptides
)

print(
    "Peptides matched in at least one sequence:",
    len(matched_peptides)
)

print(
    "Alignment cells coloured:",
    len(cell_to_peptide)
)

if unmatched_peptides:
    print(
        "Peptides not found in alignment:",
        unmatched_peptides
    )

Peptides matched in at least one sequence: 10
Alignment cells coloured: 695


In [8]:
def peptide_color_func(
    row_pos,
    col_pos,
    seq_char,
    msa
):
    peptide = cell_to_peptide.get(
        (row_pos, col_pos)
    )

    if peptide is not None:
        return peptide_colors[peptide]

    return "white"


mv = MsaViz(
    msa_file,
    format="fasta",
    wrap_length=wrap_length,
    color_scheme="None",
    show_label=True,
    label_type="id",
    show_grid=False,
    show_count=True,
    show_consensus=False,
    sort=False,
)

mv.set_custom_color_func(peptide_color_func)

mv.set_plot_params(
    ticks_interval=5,
    x_unit_size=x_unit_size,
    y_unit_size=y_unit_size,
    show_consensus_char=False,
)

fig = mv.plotfig(dpi=600)
fig.set_size_inches(
    16,   # width in inches
    8     # height in inches
)
n_sequences = len(alignment)

msa_axes = [
    ax
    for ax in fig.axes
    if abs(ax.get_ylim()[0]) < 1e-8
    and abs(ax.get_ylim()[1] - n_sequences) < 1e-8
]

if not msa_axes:
    raise RuntimeError(
        "Could not identify the alignment plotting axes."
    )

amino_acids = set(
    "ACDEFGHIKLMNPQRSTVWY-."
)

# Enlarge residues; rename and colour sequence labels by species.
for ax in msa_axes:
    x_min, _ = ax.get_xlim()
    label_x = x_min - 1

    for text in ax.texts:
        text_content = text.get_text()
        x_position, y_position = text.get_position()

        is_sequence_label = (
            abs(x_position - label_x) < 1e-8
        )

        if is_sequence_label:
            row_pos = (
                n_sequences
                - 1
                - int(round(y_position - 0.5))
            )

            if 0 <= row_pos < n_sequences:
                species = sequence_species[row_pos]

                text.set_text(
                    sequence_labels[row_pos]
                )
                text.set_color(
                    species_colors[species]
                )
                text.set_fontweight("bold")
                text.set_fontsize(9)

        elif (
            len(text_content) == 1
            and text_content in amino_acids
        ):
            text.set_fontsize(residue_font_size)


# Add one PTM marker per site × modification class.
# PTMs at the same residue are stacked vertically.
unplaced_markers = []

for _, marker_row in ptm_marker_table.iterrows():
    ptm_type = marker_row["modification_type"]
    style = marker_styles[ptm_type]

    # One-based alignment column N is centred at N - 0.5.
    x_position = (
        float(marker_row["alignment_column"])
        - 0.5
    )

    placed = False

    for ax in msa_axes:
        x_min, x_max = ax.get_xlim()

        if x_min <= x_position < x_max:
            ax.plot(
                x_position,
                n_sequences
                + 0.45
                + float(marker_row["y_offset"]),
                marker=style["marker"],
                color=style["color"],
                markersize=style["size"],
                linestyle="None",
                clip_on=False,
                zorder=20,
            )

            placed = True
            break

    if not placed:
        unplaced_markers.append(
            marker_row["canonical_site"]
        )

if unplaced_markers:
    raise RuntimeError(
        "PTM markers could not be placed: "
        f"{sorted(set(unplaced_markers))}"
    )


fig

In [9]:
fig.savefig(
    out_dir / f"{dataset_tag}_identified_peptides_and_PTM_sites.pdf",
    bbox_inches="tight",
    pad_inches=0.15,
)

fig.savefig(
    out_dir / f"{dataset_tag}_identified_peptides_and_PTM_sites.tiff",
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.15,
)

In [10]:
ptm_legend_handles = [
    Line2D(
        [0],
        [0],
        marker=marker_styles[ptm_type]["marker"],
        color="black",
        linestyle="None",
        markersize=marker_styles[ptm_type]["size"] + 2,
        label=ptm_labels[ptm_type],
    )
    for ptm_type in used_ptm_types
]

ptm_ncol = min(6, len(ptm_legend_handles))
ptm_nrow = math.ceil(
    len(ptm_legend_handles) / ptm_ncol
)

ptm_legend_fig = plt.figure(
    figsize=(10, 0.9 + 0.45 * ptm_nrow)
)

ptm_legend_fig.legend(
    handles=ptm_legend_handles,
    loc="center",
    ncol=ptm_ncol,
    frameon=False,
    title="PTMs",
)

ptm_legend_fig.savefig(
    out_dir / f"{dataset_tag}_PTM_symbol_legend.pdf",
    bbox_inches="tight",
    pad_inches=0.1,
)

ptm_legend_fig.savefig(
    out_dir / f"{dataset_tag}_PTM_symbol_legend.tiff",
    dpi=1200,
    bbox_inches="tight",
    pad_inches=0.1,
)

ptm_legend_fig

<Figure size 1000x180 with 0 Axes>

<Figure size 1000x180 with 0 Axes>

In [4]:
session_info.show()